# DuoSentimen — Analisis Sentimen Ulasan Duolingo

> **Kerangka Kerja:**  
> Dataset Ulasan Duolingo → Case Folding → Cleaning → Tokenizing → Stopword Removal → Stemming  
> → Pembobotan TF-IDF → Pembagian Data 80:20 → Model Naive Bayes  
> → Klasifikasi Sentimen (Positif / Negatif / Netral) → Evaluasi Model

**Labeling Sentimen (Rating-based):**
- ⭐⭐⭐⭐⭐ (4–5) → **POSITIF**
- ⭐⭐⭐ (3) → **NETRAL**
- ⭐⭐ (1–2) → **NEGATIF**

## 0. Instalasi Library yang Dibutuhkan

In [ ]:
# Install library yang diperlukan
import subprocess, sys

packages = [
    'google-play-scraper',
    'PySastrawi',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'wordcloud',
    'nltk',
    'openpyxl',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Semua library berhasil diinstall!')

## 1. Import Library

In [ ]:
import re
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime

# NLP
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# Word Cloud
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('✅ Import library selesai!')
print(f'   pandas     : {pd.__version__}')
print(f'   numpy      : {np.__version__}')
print(f'   sklearn    : 1.x')

---
## 2. Scraping Data Ulasan Duolingo dari Google Play Store

In [ ]:
from google_play_scraper import reviews, Sort

APP_ID    = 'com.duolingo'   # ID aplikasi Duolingo di Play Store
LANG      = 'id'             # Bahasa Indonesia
COUNTRY   = 'id'             # Negara Indonesia
MAX_REVIEWS = 2000           # Jumlah ulasan yang ingin diambil

print(f'📥 Memulai scraping {MAX_REVIEWS} ulasan dari Play Store...')
print(f'   App ID  : {APP_ID}')
print(f'   Bahasa  : {LANG} | Negara: {COUNTRY}')
print()

all_reviews = []
continuation_token = None

while len(all_reviews) < MAX_REVIEWS:
    batch_size = min(200, MAX_REVIEWS - len(all_reviews))
    try:
        result, continuation_token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_size,
            continuation_token=continuation_token,
        )
        all_reviews.extend(result)
        print(f'   Progress: {len(all_reviews)}/{MAX_REVIEWS} ulasan...', end='\r')
        if not continuation_token:
            break
    except Exception as e:
        print(f'\n⚠️  Error saat scraping: {e}')
        break

print(f'\n✅ Scraping selesai! Total: {len(all_reviews)} ulasan')

In [ ]:
# Konversi ke DataFrame
df_raw = pd.DataFrame([
    {
        'review_id'   : r.get('reviewId', ''),
        'username'    : r.get('userName', ''),
        'rating'      : r.get('score', None),
        'review_text' : r.get('content', ''),
        'app_version' : r.get('appVersion', ''),
        'review_date' : r.get('at', None),
        'thumbs_up'   : r.get('thumbsUpCount', 0),
    }
    for r in all_reviews
])

print(f'Shape dataset : {df_raw.shape}')
print(f'Distribusi rating :')
print(df_raw['rating'].value_counts().sort_index())
df_raw.head()

---
## 3. Labeling Sentimen (Rating-Based)

Sesuai dengan pendekatan **rating-based labeling**:

| Rating | Label |
|--------|-------|
| 4 – 5  | POSITIF |
| 3      | NETRAL  |
| 1 – 2  | NEGATIF |

In [ ]:
def label_from_rating(rating):
    """Konversi rating bintang ke label sentimen."""
    if rating is None:
        return None
    if rating >= 4:
        return 'POSITIF'
    elif rating == 3:
        return 'NETRAL'
    else:
        return 'NEGATIF'

df_raw['sentiment_label'] = df_raw['rating'].apply(label_from_rating)

# Hapus baris tanpa teks atau label
df = df_raw.dropna(subset=['review_text', 'sentiment_label']).copy()
df = df[df['review_text'].str.strip() != ''].reset_index(drop=True)

print('\n📊 Distribusi Label Sentimen:')
label_counts = df['sentiment_label'].value_counts()
for label, count in label_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:10s}: {count:5d} ({pct:.1f}%) {bar}')
print(f'\n  Total: {len(df)} ulasan')

In [ ]:
# Visualisasi distribusi label
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

COLORS = {'POSITIF': '#2ecc71', 'NETRAL': '#f39c12', 'NEGATIF': '#e74c3c'}
labels_order = ['POSITIF', 'NETRAL', 'NEGATIF']
counts = [label_counts.get(l, 0) for l in labels_order]
colors = [COLORS[l] for l in labels_order]

# Bar chart
bars = axes[0].bar(labels_order, counts, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Label Sentimen', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label Sentimen')
axes[0].set_ylabel('Jumlah Ulasan')
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(cnt), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=labels_order, colors=colors, autopct='%1.1f%%',
            startangle=90, pctdistance=0.85)
axes[1].set_title('Proporsi Sentimen', fontsize=14, fontweight='bold')

plt.suptitle('Analisis Distribusi Sentimen Ulasan Duolingo\n(Rating-Based Labeling)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distribusi_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot distribusi sentimen tersimpan: distribusi_sentimen.png')

In [ ]:
# Distribusi Rating bintang
fig, ax = plt.subplots(figsize=(8, 5))
rating_counts = df['rating'].value_counts().sort_index()
rating_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=rating_colors, edgecolor='white', linewidth=1.5)
ax.set_title('Distribusi Rating Bintang', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating (Bintang)')
ax.set_ylabel('Jumlah Ulasan')
ax.set_xticks([1,2,3,4,5])
ax.set_xticklabels(['⭐ 1', '⭐⭐ 2', '⭐⭐⭐ 3', '⭐⭐⭐⭐ 4', '⭐⭐⭐⭐⭐ 5'])
for bar, cnt in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(cnt), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('distribusi_rating.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Preprocessing Teks

Sesuai kerangka kerja:
1. **Case Folding** — mengubah huruf menjadi huruf kecil
2. **Cleaning** — menghapus URL, mention, hashtag, emoji, simbol
3. **Tokenizing** — memecah kalimat menjadi kata
4. **Normalisasi Slang** — mengganti kata tidak baku
5. **Stopword Removal** — menghapus kata-kata umum
6. **Stemming** — mengubah ke bentuk kata dasar (PySastrawi)

In [ ]:
# ── Kamus Slang / Singkatan ──────────────────────────────────────────────────
SLANG_DICT = {
    'bgt': 'banget', 'banget': 'banget', 'gak': 'tidak', 'ga': 'tidak',
    'gk': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'enggak': 'tidak',
    'sy': 'saya', 'gw': 'saya', 'gue': 'saya', 'aku': 'saya',
    'lo': 'kamu', 'lu': 'kamu', 'yg': 'yang', 'klo': 'kalau',
    'kl': 'kalau', 'kal': 'kalau', 'dg': 'dengan', 'dgn': 'dengan',
    'utk': 'untuk', 'dr': 'dari', 'pk': 'pakai', 'tdk': 'tidak',
    'sdh': 'sudah', 'blm': 'belum', 'krn': 'karena', 'tp': 'tapi',
    'tpi': 'tapi', 'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'nih': 'ini', 'tuh': 'itu', 'sih': '', 'deh': '', 'dong': '',
    'lho': '', 'lah': '', 'wkwk': '', 'haha': '', 'hehe': '',
    'gbsa': 'tidak bisa', 'bikin': 'buat', 'mantul': 'mantap',
    'kuy': 'ayo', 'cuy': '', 'bro': '', 'mimin': 'admin',
    'min': 'admin', 'woyy': 'hei', 'woy': 'hei', 'tolong': 'tolong',
    'pliss': 'tolong', 'pls': 'tolong', 'thx': 'terima kasih',
    'makasih': 'terima kasih', 'makasi': 'terima kasih',
    'ngebug': 'error', 'lemot': 'lambat', 'lelet': 'lambat',
    'males': 'malas', 'niat': 'niat', 'asik': 'asyik',
    'mantap': 'mantap', 'keren': 'keren', 'bagus': 'bagus',
    'jelek': 'jelek', 'buruk': 'buruk', 'parah': 'parah',
    'ok': 'oke', 'oke': 'oke', 'sip': 'bagus',
}

# ── Stopwords Bahasa Indonesia ───────────────────────────────────────────────
STOPWORDS = {
    'yang', 'dan', 'di', 'ke', 'dari', 'dengan', 'untuk', 'ini', 'itu',
    'ada', 'akan', 'adalah', 'saya', 'kamu', 'mereka', 'kami', 'kita',
    'tidak', 'bisa', 'jika', 'maka', 'pada', 'atau', 'juga', 'sudah',
    'belum', 'saat', 'masih', 'lagi', 'jadi', 'seperti', 'lebih',
    'cara', 'dapat', 'harus', 'saja', 'pun', 'pula', 'bahwa',
    'satu', 'dua', 'tiga', 'empat', 'lima', 'enam', 'tujuh',
    'delapan', 'sembilan', 'sepuluh', 'sangat', 'sekali', 'agak', 'cukup',
    'kalau', 'mau', 'ya', 'iya', 'oh', 'ah', 'eh', 'hm',
    'app', 'aplikasi', 'duolingo', 'play', 'google', 'update',
    'hp', 'android', 'ios', 'nya', 'aja',
}

print('✅ Kamus slang dan stopwords dimuat!')
print(f'   Jumlah entri slang     : {len(SLANG_DICT)}')
print(f'   Jumlah kata stopwords  : {len(STOPWORDS)}')

In [ ]:
# ── Inisialisasi Stemmer PySastrawi ──────────────────────────────────────────
print('⏳ Memuat Sastrawi Stemmer...')
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print('✅ Sastrawi Stemmer siap!')

In [ ]:
# ── Fungsi Pipeline Preprocessing ───────────────────────────────────────────

def step1_case_folding(text: str) -> str:
    """Langkah 1: Case Folding — ubah semua huruf menjadi huruf kecil."""
    return text.lower()


def step2_cleaning(text: str) -> str:
    """Langkah 2: Cleaning — hapus URL, mention, hashtag, angka, simbol, emoji."""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)    # hapus URL
    text = re.sub(r'@\w+', '', text)                       # hapus mention
    text = re.sub(r'#\w+', '', text)                       # hapus hashtag
    text = text.encode('ascii', 'ignore').decode('ascii')  # hapus emoji/unicode
    text = re.sub(r'\d+', '', text)                        # hapus angka
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)              # hapus simbol
    text = re.sub(r'\s+', ' ', text).strip()               # normalisasi spasi
    return text


def step3_tokenizing(text: str) -> list:
    """Langkah 3: Tokenizing — pecah teks menjadi daftar token/kata."""
    tokens = text.split()
    return [t for t in tokens if len(t) > 1]  # hapus token 1 karakter


def step3b_normalize_slang(tokens: list) -> list:
    """Normalisasi slang/singkatan ke kata baku."""
    return [SLANG_DICT.get(t, t) for t in tokens if SLANG_DICT.get(t, t) != '']


def step4_stopword_removal(tokens: list) -> list:
    """Langkah 4: Stopword Removal — hapus kata-kata umum yang tidak bermakna."""
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def step5_stemming(tokens: list) -> list:
    """Langkah 5: Stemming — ubah setiap kata ke bentuk dasarnya."""
    return [stemmer.stem(t) for t in tokens]


def full_pipeline(raw_text: str) -> str:
    """Pipeline lengkap sesuai kerangka kerja:
    Case Folding → Cleaning → Tokenizing → Normalisasi → Stopword Removal → Stemming
    """
    text   = step1_case_folding(raw_text)       # 1. Case Folding
    text   = step2_cleaning(text)               # 2. Cleaning
    tokens = step3_tokenizing(text)             # 3. Tokenizing
    tokens = step3b_normalize_slang(tokens)     # 4. Normalisasi slang
    tokens = step4_stopword_removal(tokens)     # 5. Stopword Removal
    tokens = step5_stemming(tokens)             # 6. Stemming
    return ' '.join(tokens)


# Demo pipeline
contoh = 'Aplikasi ini bgt keren bgt! Tapi sering ngebug dan lemot, min tolong fix dong!!'
print('🔍 Demo Pipeline Preprocessing:')
print(f'  Raw        : {contoh}')
print(f'  Case Fold  : {step1_case_folding(contoh)}')
_cf = step1_case_folding(contoh)
_cl = step2_cleaning(_cf)
print(f'  Cleaning   : {_cl}')
_tk = step3_tokenizing(_cl)
print(f'  Tokenizing : {_tk}')
_nm = step3b_normalize_slang(_tk)
print(f'  Normalisasi: {_nm}')
_sw = step4_stopword_removal(_nm)
print(f'  Stopword   : {_sw}')
_st = step5_stemming(_sw)
print(f'  Stemming   : {_st}')
print(f'  Hasil Akhir: {full_pipeline(contoh)}')

In [ ]:
# Terapkan pipeline ke seluruh dataset
print('⏳ Menjalankan preprocessing pipeline ke seluruh dataset...')
print(f'   Total ulasan: {len(df)}')

df['cleaned_text'] = df['review_text'].progress_apply(full_pipeline) if hasattr(df['review_text'], 'progress_apply') else df['review_text'].apply(full_pipeline)

# Hapus hasil kosong setelah preprocessing
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)

print(f'✅ Preprocessing selesai!')
print(f'   Dataset final: {len(df)} ulasan')
df[['review_text', 'cleaned_text', 'rating', 'sentiment_label']].head(10)

In [ ]:
# Statistik hasil preprocessing
df['word_count_raw']     = df['review_text'].apply(lambda x: len(x.split()))
df['word_count_cleaned'] = df['cleaned_text'].apply(lambda x: len(x.split()))

print('📊 Statistik Preprocessing:')
stats = df[['word_count_raw', 'word_count_cleaned']].describe()
stats.columns = ['Kata (Raw)', 'Kata (Cleaned)']
print(stats.round(2).to_string())
print(f'\n   Rata-rata pengurangan kata: {(df["word_count_raw"].mean() - df["word_count_cleaned"].mean()):.1f} kata/ulasan')

---
## 5. Word Cloud per Kelas Sentimen

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

LABEL_INFO = [
    ('POSITIF', '#2ecc71', '⭐⭐⭐⭐⭐ Positif'),
    ('NETRAL',  '#f39c12', '⭐⭐⭐ Netral'),
    ('NEGATIF', '#e74c3c', '⭐⭐ Negatif'),
]

for ax, (label, color, title) in zip(axes, LABEL_INFO):
    texts = ' '.join(df[df['sentiment_label'] == label]['cleaned_text'].tolist())
    if texts.strip():
        wc = WordCloud(
            width=600, height=400,
            background_color='white',
            colormap='Greens' if label == 'POSITIF' else ('Oranges' if label == 'NETRAL' else 'Reds'),
            max_words=80,
            prefer_horizontal=0.9,
        ).generate(texts)
        ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.axis('off')

plt.suptitle('Word Cloud per Kelas Sentimen', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word cloud tersimpan: wordcloud_sentimen.png')

---
## 6. Pembobotan TF-IDF & Pembagian Data 80:20

In [ ]:
# ── Data dan Label ───────────────────────────────────────────────────────────
X = df['cleaned_text'].values
y = df['sentiment_label'].values

print(f'📦 Total data siap training : {len(X)}')
print(f'   Distribusi kelas         :')
unique, counts = np.unique(y, return_counts=True)
for lbl, cnt in zip(unique, counts):
    print(f'     {lbl:10s}: {cnt} ({cnt/len(y)*100:.1f}%)')

# ── Pembagian Data 80:20 ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'\n✂️  Pembagian Data 80:20:')
print(f'   Training set : {len(X_train)} data ({len(X_train)/len(X)*100:.1f}%)')
print(f'   Testing set  : {len(X_test)} data ({len(X_test)/len(X)*100:.1f}%)')

In [ ]:
# Visualisasi pembagian data
fig, ax = plt.subplots(figsize=(8, 4))

splits = ['Training (80%)', 'Testing (20%)']
sizes  = [len(X_train), len(X_test)]
colors = ['#3498db', '#e67e22']

bars = ax.barh(splits, sizes, color=colors, edgecolor='white', height=0.5)
for bar, size in zip(bars, sizes):
    ax.text(bar.get_width() / 2, bar.get_y() + bar.get_height()/2,
            f'{size} data', ha='center', va='center',
            color='white', fontweight='bold', fontsize=12)

ax.set_title('Pembagian Dataset 80:20', fontsize=14, fontweight='bold')
ax.set_xlabel('Jumlah Data')
ax.set_xlim(0, len(X) * 1.1)
plt.tight_layout()
plt.savefig('pembagian_data.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Model Naive Bayes dengan TF-IDF

Sesuai kerangka kerja: **TF-IDF Vectorizer** + **Multinomial Naive Bayes**

In [ ]:
# ── Bangun Pipeline: TF-IDF + Naive Bayes ───────────────────────────────────
print('🚀 Membangun model Naive Bayes dengan TF-IDF...')

nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),    # unigram dan bigram
        max_features=20000,    # maksimal 20.000 fitur
        min_df=2,              # minimal muncul di 2 dokumen
        sublinear_tf=True,     # gunakan log(tf)
    )),
    ('nb', MultinomialNB(alpha=1.0)),  # alpha=1 (Laplace smoothing)
])

# Training
print('⏳ Training model...')
nb_pipeline.fit(X_train, y_train)
print('✅ Model selesai ditraining!')

# Info TF-IDF
tfidf = nb_pipeline.named_steps['tfidf']
print(f'\n📊 Info TF-IDF:')
print(f'   Jumlah fitur/vocabulary : {len(tfidf.vocabulary_):,}')
print(f'   Ngram range             : {tfidf.ngram_range}')

In [ ]:
# Top fitur penting per kelas
feature_names = tfidf.get_feature_names_out()
nb_clf = nb_pipeline.named_steps['nb']
classes = nb_clf.classes_

print('🔑 Top 15 kata penting per kelas:')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
bar_colors = ['#2ecc71', '#f39c12', '#e74c3c']

for idx, (cls, color) in enumerate(zip(classes, bar_colors)):
    log_probs = nb_clf.feature_log_prob_[idx]
    top_indices = np.argsort(log_probs)[-15:]
    top_features = feature_names[top_indices]
    top_probs = log_probs[top_indices]
    
    axes[idx].barh(top_features, top_probs, color=color, alpha=0.85)
    axes[idx].set_title(f'Kelas: {cls}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Log Probability')

plt.suptitle('Top 15 Fitur per Kelas Sentimen (TF-IDF + Naive Bayes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('top_fitur.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Evaluasi Model

**Metrik:** Accuracy, Precision, Recall, F1-Score

In [ ]:
# Prediksi pada data testing
y_pred = nb_pipeline.predict(X_test)

# ── Hitung Metrik Evaluasi ───────────────────────────────────────────────────
CLASSES = ['POSITIF', 'NEGATIF', 'NETRAL']

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, labels=CLASSES, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, labels=CLASSES, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred, labels=CLASSES, average='weighted', zero_division=0)

print('=' * 55)
print('       HASIL EVALUASI MODEL NAIVE BAYES')
print('=' * 55)
print(f'  Accuracy  (Akurasi)    : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precision (Presisi)    : {precision:.4f} ({precision*100:.2f}%)')
print(f'  Recall    (Sensitivitas): {recall:.4f} ({recall*100:.2f}%)')
print(f'  F1-Score               : {f1:.4f} ({f1*100:.2f}%)')
print('=' * 55)
print(f'  Data training : {len(X_train)} | Data testing: {len(X_test)}')
print('=' * 55)

In [ ]:
# Classification Report lengkap per kelas
print('\n📋 Classification Report Detail per Kelas:')
print(classification_report(y_test, y_pred, target_names=CLASSES, zero_division=0))

In [ ]:
# ── Visualisasi Metrik Evaluasi ──────────────────────────────────────────────
metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]
metric_colors  = ['#3498db', '#9b59b6', '#2ecc71', '#e67e22']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart metrik
bars = axes[0].bar(metrics_names, metrics_values, color=metric_colors,
                   edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Metrik Evaluasi Model Naive Bayes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score')
for bar, val in zip(bars, metrics_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target 80%')
axes[0].legend()

# Radar chart
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist()
values = metrics_values + [metrics_values[0]]
angles += angles[:1]

ax_radar = axes[1]
ax_radar = plt.subplot(122, polar=True)
ax_radar.plot(angles, values, 'o-', linewidth=2, color='#3498db')
ax_radar.fill(angles, values, alpha=0.25, color='#3498db')
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics_names, fontsize=11)
ax_radar.set_ylim(0, 1)
ax_radar.set_title('Radar Chart Evaluasi', fontsize=13, fontweight='bold', pad=20)
ax_radar.yaxis.set_tick_params(labelsize=8)

plt.suptitle('Evaluasi Model: Accuracy | Precision | Recall | F1-Score',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot evaluasi tersimpan: evaluasi_model.png')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix angka
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Jumlah)', fontsize=13, fontweight='bold')

# Confusion matrix persentase
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('Confusion Matrix (Persentase %)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.suptitle('Confusion Matrix — Model Naive Bayes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix tersimpan: confusion_matrix.png')

---
## 9. Simpan Dataset & Model

In [ ]:
import joblib
import os

os.makedirs('output', exist_ok=True)

# Simpan dataset lengkap
df_export = df[['review_text', 'cleaned_text', 'rating', 'sentiment_label',
                'review_date', 'app_version', 'thumbs_up']].copy()
df_export.to_csv('output/dataset_duolingo_ulasan.csv', index=False, encoding='utf-8-sig')
df_export.to_excel('output/dataset_duolingo_ulasan.xlsx', index=False)
print(f'✅ Dataset tersimpan:')
print(f'   output/dataset_duolingo_ulasan.csv  ({len(df_export)} baris)')
print(f'   output/dataset_duolingo_ulasan.xlsx ({len(df_export)} baris)')

# Simpan model
joblib.dump(nb_pipeline, 'output/model_naive_bayes_tfidf.pkl')
print(f'\n✅ Model tersimpan: output/model_naive_bayes_tfidf.pkl')

# Simpan hasil metrik
hasil_metrik = pd.DataFrame([{
    'Algorithm'    : 'Multinomial Naive Bayes',
    'Vectorizer'   : 'TF-IDF (unigram+bigram)',
    'Train Size'   : len(X_train),
    'Test Size'    : len(X_test),
    'Accuracy'     : round(accuracy, 4),
    'Precision'    : round(precision, 4),
    'Recall'       : round(recall, 4),
    'F1-Score'     : round(f1, 4),
    'Timestamp'    : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}])
hasil_metrik.to_csv('output/hasil_evaluasi_model.csv', index=False)
hasil_metrik.to_excel('output/hasil_evaluasi_model.xlsx', index=False)
print(f'\n✅ Hasil evaluasi tersimpan: output/hasil_evaluasi_model.csv')
print('\n' + '='*55)
print('   RINGKASAN AKHIR')
print('='*55)
print(hasil_metrik.T.to_string(header=False))

---
## 10. Uji Prediksi Manual

In [ ]:
def predict_sentiment(text: str) -> dict:
    """Prediksi sentimen untuk teks baru."""
    cleaned = full_pipeline(text)
    label   = nb_pipeline.predict([cleaned])[0]
    proba   = nb_pipeline.predict_proba([cleaned])[0]
    classes = nb_pipeline.classes_
    proba_dict = dict(zip(classes, proba))
    return {
        'input'    : text,
        'cleaned'  : cleaned,
        'label'    : label,
        'confidence': round(max(proba) * 100, 2),
        'proba'    : {k: round(v*100, 2) for k, v in proba_dict.items()},
    }

# Uji dengan beberapa kalimat contoh
test_sentences = [
    'Aplikasi ini sangat bagus dan membantu belajar bahasa inggris!',
    'Sering crash dan error terus, tolong perbaiki segera!',
    'Biasa saja, tidak ada yang spesial.',
    'Mantap banget! Streak 200 hari tercapai, terima kasih Duolingo!',
    'Lambat banget loadingnya, boros kuota, jelek!',
    'Lumayan bagus untuk belajar tapi ada beberapa bug kecil.',
]

print('🔮 Hasil Prediksi Manual:')
print('='*70)
EMOJI = {'POSITIF': '😊 ✅', 'NETRAL': '😐 ➖', 'NEGATIF': '😠 ❌'}
for sent in test_sentences:
    result = predict_sentiment(sent)
    print(f"Input      : {result['input']}")
    print(f"Label      : {EMOJI.get(result['label'], '')} {result['label']} ({result['confidence']}% confidence)")
    print(f"Probabilitas: {result['proba']}")
    print('-'*70)

---
## Ringkasan Kerangka Kerja yang Diimplementasikan

| Tahap | Metode | Status |
|-------|--------|--------|
| Dataset | Ulasan Duolingo (Google Play ID) | ✅ |
| Labeling | Rating-Based (4-5=Positif, 3=Netral, 1-2=Negatif) | ✅ |
| Case Folding | `str.lower()` | ✅ |
| Cleaning | Regex (URL, mention, emoji, simbol) | ✅ |
| Tokenizing | `str.split()` | ✅ |
| Normalisasi | Kamus Slang Bahasa Indonesia | ✅ |
| Stopword Removal | Daftar stopwords Indonesia | ✅ |
| Stemming | PySastrawi | ✅ |
| Pembobotan | TF-IDF (unigram+bigram, max 20.000 fitur) | ✅ |
| Pembagian Data | 80% Training / 20% Testing (stratified) | ✅ |
| Model | Multinomial Naive Bayes (Laplace smoothing α=1) | ✅ |
| Klasifikasi | POSITIF / NEGATIF / NETRAL | ✅ |
| Evaluasi | Accuracy, Precision, Recall, F1-Score | ✅ |